### Big Data | CS-GY 6513 C

### Title : Graph-Based Analysis of Academic Citation Networks                                                                 
   
### Team Members :                                                                                                       
                                                                                                                      
  | Name | NetID |                                                                                                      
  |------|-------|
  | Navya Gupta | NG3118 |                                                                                              
  | Hardik Amarwani | HA3290 |                                                                                        
  | Shreyansh Saurabh | SS21034 |

### Environment setup

In [1]:
pip install pyspark graphframes pyarrow ipywidgets pyvis==0.3.1 networkx matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — make sure Python can see your module, and import the bits you need
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / "modules").is_dir() and (candidate / "driver.ipynb").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODULE_DIR = PROJECT_ROOT / "modules"
sys.path.append(str(MODULE_DIR))
print(MODULE_DIR)

import keyword_search_module as ksm
import publication_search_module as psm

# bring functions into local namespace if you like
from keyword_search_module import initialize_spark, search_papers_widget, build_graph_widget, build_graph_from_keywords
from publication_search_module import build_publication_graph_widget

/Users/shreyanshsaurabh/Desktop/BIGDATA/Big-Data-Project-Research-Net/modules


In [3]:
# Cell 3 — start Spark and distribute our module (so that ksm.spark & ksm.sc will be available, and set PROJECT_ROOT)
import os
import sys
from pathlib import Path
import keyword_search_module  # notebook driver can already see it

# force the same Python for driver and workers
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# initialize Spark
spark, sc = initialize_spark(driver_memory="6g", shuffle_partitions=32)

# ship the module to executors
MODULE_DIR = PROJECT_ROOT / "modules"
MODULE_FILE = MODULE_DIR / "keyword_search_module.py"
MODULE_FILE2 = MODULE_DIR / "publication_search_module.py"
sc.addPyFile(str(MODULE_FILE))
sc.addPyFile(str(MODULE_FILE2))

# set up PROJECT_ROOT and checkpointing
PROJECT_ROOT_KEYWORD_SEARCH = PROJECT_ROOT / "keyword_search"
PROJECT_ROOT_KEYWORD_SEARCH.mkdir(exist_ok=True, parents=True)
sc.setCheckpointDir(str(PROJECT_ROOT_KEYWORD_SEARCH / "checkpoints"))

PROJECT_ROOT_PUBLICATION_SEARCH = PROJECT_ROOT / "publication_search"
PROJECT_ROOT_PUBLICATION_SEARCH.mkdir(exist_ok=True, parents=True)
# (optional) sc.setCheckpointDir(str(PROJECT_ROOT_PUBLICATION_SEARCH/"checkpoints"))
psm.PROJECT_ROOT = PROJECT_ROOT_PUBLICATION_SEARCH

# make PROJECT_ROOT available inside the module too
import keyword_search_module as m; m.PROJECT_ROOT = PROJECT_ROOT_KEYWORD_SEARCH

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/01 16:30:14 WARN Utils: Your hostname, SS---Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.188 instead (on interface en0)
26/05/01 16:30:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/homebrew/Cellar/jupyterlab/4.4.10/libexec/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/shreyanshsaurabh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/shreyanshsaurabh/.ivy2.5.2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-31556afb-fa97-4c6a-8125-28b6a116b143;1.0
	confs: [default]
	found graphframes#graphframes;0.8.4-spark3.5-s_2.13 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 553ms :: artifacts dl 32ms
	:: m

In [4]:
# Cell 5: import and distribute paper-ID module
import sys
from pathlib import Path
# set up PROJECT_ROOT and checkpointing

PROJECT_ROOT_PAPER_ID_SEARCH = PROJECT_ROOT / "paper_id_search"
PROJECT_ROOT_PAPER_ID_SEARCH.mkdir(exist_ok=True, parents=True)
sc.setCheckpointDir(str(PROJECT_ROOT_PAPER_ID_SEARCH / "checkpoints"))

MODULE_DIR = PROJECT_ROOT / "modules"
MODULE_FILE_PAPER_ID_SEARCH = MODULE_DIR / "paper_id_search_module.py"
sc.addPyFile(str(MODULE_FILE_PAPER_ID_SEARCH))

import paper_id_search_module as pidm
from paper_id_search_module import build_id_graph_widget
pidm.PROJECT_ROOT = PROJECT_ROOT_PAPER_ID_SEARCH

### Widget for Keyword based search

In [5]:
# Cell 4 — build & visualize the graph via the module’s widget UI
build_graph_widget(spark, sc)

### Widget for paper id search

In [6]:
# Cell 6: show the graph-builder widget
build_id_graph_widget(spark, sc)

### Widget for Publication and year based search

In [7]:
# Cell 7 — build & visualize the graph via your new publication_search_module
build_publication_graph_widget(spark, sc)